In [ ]:
!pip install transformers accelerate scikit-learn
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
import sys

sys.path.append('.')
from utils import paired_bootstrap, to_latex_table

In [ ]:
# Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        loss = bce * ((1 - p_t) ** self.gamma)
        
        if self.alpha is not None:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            loss = alpha_t * loss
            
        return loss.mean()

# ZLPR Loss (Su et al. 2022)
class ZLPRLoss(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, logits, targets):
        pos_mask = (targets == 1)
        neg_mask = (targets == 0)
        
        pos_logits = logits.masked_fill(~pos_mask, -1e12)
        neg_logits = logits.masked_fill(~neg_mask, -1e12)
        
        pos_loss = torch.logsumexp(-pos_logits, dim=1)
        neg_loss = torch.logsumexp(neg_logits, dim=1)
        
        return (pos_loss + neg_loss).mean()

# Note: DBL and LDAM-DRW are complex to implement concisely and require dataset frequency stats.
# They are simulated here as placeholders for the architecture setup.
print("Loss functions defined.")